In [5]:
"""
Catalyst Optimizer Examples — PySpark 4.0
==========================================
Each section isolates one optimizer rule and uses explain() to surface it.

explain() modes used:
  "formatted"  — readable physical plan + optimized logical plan
  "extended"   — all four plan stages (section 7)
"""

import os
import tempfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F




In [2]:
# ── helpers ───────────────────────────────────────────────────────────────────

def section(title: str) -> None:
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}\n")


# ── session ───────────────────────────────────────────────────────────────────

spark = (
    SparkSession.builder
    .appName("CatalystOptimizerExamples")
    .master("local[*]")
    .config("spark.sql.autoBroadcastJoinThreshold", "1mb")
    .config("spark.sql.shuffle.partitions", "4")   # keep plans short
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 23:51:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 1. PREDICATE PUSHDOWN
#
#  Filters are moved as close to the data source as possible — they execute
#  before joins and aggregations so fewer rows flow upward through the plan.
# ═══════════════════════════════════════════════════════════════════════════ #
section("1. PREDICATE PUSHDOWN")
print("""
  We write:  orders.join(customers).filter(amount > 40)
  Optimizer pushes `amount > 40` BELOW the join so fewer rows are shuffled.

  Look for:  Filter node appearing INSIDE (below) the join in the physical plan.
""")

orders = spark.createDataFrame(
    [(1, 101, 50.0), (2, 102, 30.0), (3, 101, 80.0), (4, 103, 20.0)],
    ["order_id", "customer_id", "amount"],
)
customers = spark.createDataFrame(
    [(101, "Alice"), (102, "Bob"), (103, "Charlie")],
    ["customer_id", "name"],
)

orders.join(customers, "customer_id").filter(F.col("amount") > 40).explain("formatted")



  1. PREDICATE PUSHDOWN


  We write:  orders.join(customers).filter(amount > 40)
  Optimizer pushes `amount > 40` BELOW the join so fewer rows are shuffled.

  Look for:  Filter node appearing INSIDE (below) the join in the physical plan.

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [3]: [order_id#0L, customer_id#1L, amount#2]
Arguments: [order_id#0L, customer_id#1L, amount#2], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [order_id#0L, customer_id#1L, amount#2]
Condition : ((isnotnull(amount#2) AND (amount#2 > 40.0)) AND isnotnull(customer_id#1L))

(3) Exchange
Input [3]: [order_id#0L, customer_id#1L, am

In [7]:

# ═══════════════════════════════════════════════════════════════════════════ #
# 2. COLUMN PRUNING  (Projection Pushdown)
#
#  Columns never referenced downstream are dropped at the scan level —
#  they are never read from storage or carried through the plan.
# ═══════════════════════════════════════════════════════════════════════════ #
section("2. COLUMN PRUNING (Projection Pushdown)")
print("""
  We create a 6-column DataFrame but only select 2 columns downstream.
  The other 4 columns (b, c, d, e) should never appear in the physical plan.

  Look for:  The scan / Project listing ONLY `id` and `a`.
""")

wide = spark.range(100).select(
    F.col("id"),
    F.col("id").alias("a"),
    F.col("id").alias("b"),   # unused
    F.col("id").alias("c"),   # unused
    F.col("id").alias("d"),   # unused
    F.col("id").alias("e"),   # unused
)
wide.select("id", "a").filter(F.col("id") < 50).explain("formatted")




  2. COLUMN PRUNING (Projection Pushdown)


  We create a 6-column DataFrame but only select 2 columns downstream.
  The other 4 columns (b, c, d, e) should never appear in the physical plan.

  Look for:  The scan / Project listing ONLY `id` and `a`.

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#5L]
Arguments: Range (0, 100, step=1, splits=Some(10))

(2) Filter [codegen id : 1]
Input [1]: [id#5L]
Condition : (id#5L < 50)

(3) Project [codegen id : 1]
Output [2]: [id#5L, id#5L AS a#6L]
Input [1]: [id#5L]




In [8]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 3. CONSTANT FOLDING
#
#  Expressions containing only literals are evaluated once at compile time,
#  not for every row at runtime.
# ═══════════════════════════════════════════════════════════════════════════ #
section("3. CONSTANT FOLDING")
print("""
  `id > (10 - 5)` and `id > 5` should produce identical physical plans.
  The subtraction is folded to 5 during the Optimizer phase.

  Look for:  both plans showing the same literal (5) in the Filter node.
""")

df = spark.range(20)

print("--- filter: id > (10 - 5)  [constant folded] ---")
df.filter(F.col("id") > (F.lit(10) - F.lit(5))).explain("formatted")

print("--- filter: id > 5  [already a literal — same plan] ---")
df.filter(F.col("id") > 5).explain("formatted")



  3. CONSTANT FOLDING


  `id > (10 - 5)` and `id > 5` should produce identical physical plans.
  The subtraction is folded to 5 during the Optimizer phase.

  Look for:  both plans showing the same literal (5) in the Filter node.

--- filter: id > (10 - 5)  [constant folded] ---
== Physical Plan ==
* Filter (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#11L]
Arguments: Range (0, 20, step=1, splits=Some(10))

(2) Filter [codegen id : 1]
Input [1]: [id#11L]
Condition : (id#11L > 5)


--- filter: id > 5  [already a literal — same plan] ---
== Physical Plan ==
* Filter (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#11L]
Arguments: Range (0, 20, step=1, splits=Some(10))

(2) Filter [codegen id : 1]
Input [1]: [id#11L]
Condition : (id#11L > 5)




In [9]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 4. BROADCAST HASH JOIN  vs  SORT-MERGE JOIN
#
#  When one side of a join is small enough (autoBroadcastJoinThreshold),
#  Spark ships it to every executor instead of shuffling both sides.
# ═══════════════════════════════════════════════════════════════════════════ #
section("4. BROADCAST HASH JOIN vs SORT-MERGE JOIN")
print("""
  autoBroadcastJoinThreshold = 1 MB (set above).
  The `roles` table (3 rows) is far below that threshold.

  Look for:  BroadcastHashJoin  when broadcast is ON  (no Exchange on small side)
             SortMergeJoin      when broadcast is OFF  (Exchange on both sides)
""")

users = (
    spark.range(100_000)
    .withColumnRenamed("id", "user_id")
    .withColumn("role_id", (F.col("user_id") % 3 + 1).cast("int"))
)
roles = spark.createDataFrame(
    [(1, "admin"), (2, "editor"), (3, "viewer")],
    ["role_id", "role_name"],
)

print("--- BroadcastHashJoin  (threshold = 1 MB) ---")
users.join(roles, "role_id").explain("formatted")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("--- SortMergeJoin  (broadcast disabled) ---")
users.join(roles, "role_id").explain("formatted")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(1 * 1024 * 1024))



  4. BROADCAST HASH JOIN vs SORT-MERGE JOIN


  autoBroadcastJoinThreshold = 1 MB (set above).
  The `roles` table (3 rows) is far below that threshold.

  Look for:  BroadcastHashJoin  when broadcast is ON  (no Exchange on small side)
             SortMergeJoin      when broadcast is OFF  (Exchange on both sides)

--- BroadcastHashJoin  (threshold = 1 MB) ---
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin Inner BuildLeft (7)
      :- BroadcastExchange (4)
      :  +- Project (3)
      :     +- Filter (2)
      :        +- Range (1)
      +- Filter (6)
         +- Scan ExistingRDD (5)


(1) Range
Output [1]: [id#18L]
Arguments: Range (0, 100000, step=1, splits=Some(10))

(2) Filter
Input [1]: [id#18L]
Condition : isnotnull(cast(((id#18L % 3) + 1) as int))

(3) Project
Output [2]: [id#18L AS user_id#19L, cast(((id#18L % 3) + 1) as int) AS role_id#20]
Input [1]: [id#18L]

(4) BroadcastExchange
Input [2]: [user_id#19L, role_id#20]
Arguments: HashedRelati

In [11]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 5. PARTITION PRUNING
#
#  When Parquet/ORC data is written with partitionBy(), a filter on the
#  partition column causes Spark to skip entire directories at the file-system
#  level — irrelevant partitions are never opened.
# ═══════════════════════════════════════════════════════════════════════════ #
section("5. PARTITION PRUNING")
print("""
  Write 1 000 rows partitioned by `region` (5 distinct values → 5 directories).
  Read back with filter region == '2'.

  Look for:  PartitionFilters: [isnotnull(region#…), (region#… = 2)]
             in the FileScan node.  Only 1 of 5 partitions is read.
""")

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "events")
    (
        spark.range(1_000)
        .withColumn("region", (F.col("id") % 5).cast("string"))
        .write.partitionBy("region").parquet(path)
    )
    spark.read.parquet(path).filter(F.col("region") == "2").explain("formatted")



  5. PARTITION PRUNING


  Write 1 000 rows partitioned by `region` (5 distinct values → 5 directories).
  Read back with filter region == '2'.

  Look for:  PartitionFilters: [isnotnull(region#…), (region#… = 2)]
             in the FileScan node.  Only 1 of 5 partitions is read.

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [id#32L, region#33]
Batched: true
Location: InMemoryFileIndex [file:/tmp/tmpmp7ftpre/events]
PartitionFilters: [isnotnull(region#33), (region#33 = 2)]
ReadSchema: struct<id:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [id#32L, region#33]




In [12]:

# ═══════════════════════════════════════════════════════════════════════════ #
# 6. FILTER SIMPLIFICATION
#
#  Always-true filters are removed from the plan entirely.
#  Always-false filters collapse the whole plan to an empty LocalRelation
#  — no scan, no shuffle, no work at all.
# ═══════════════════════════════════════════════════════════════════════════ #
section("6. FILTER SIMPLIFICATION (always-true / always-false)")
print("""
  lit(True)  — filter is a no-op; it disappears from the plan.
  lit(False) — nothing can ever pass; the plan becomes an empty LocalRelation.
""")

base = spark.range(10)

print("--- filter(lit(True))  →  Filter node should be absent ---")
base.filter(F.lit(True)).explain("formatted")

print("--- filter(lit(False)) →  should become LocalRelation (empty, no scan) ---")
base.filter(F.lit(False)).explain("formatted")



  6. FILTER SIMPLIFICATION (always-true / always-false)


  lit(True)  — filter is a no-op; it disappears from the plan.
  lit(False) — nothing can ever pass; the plan becomes an empty LocalRelation.

--- filter(lit(True))  →  Filter node should be absent ---
== Physical Plan ==
* Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#35L]
Arguments: Range (0, 10, step=1, splits=Some(10))


--- filter(lit(False)) →  should become LocalRelation (empty, no scan) ---
== Physical Plan ==
LocalTableScan (1)


(1) LocalTableScan
Output [1]: [id#35L]
Arguments: <empty>, [id#35L]




In [13]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 7. ALL FOUR CATALYST PLAN STAGES   explain("extended")
#
#  The full Catalyst pipeline:
#   1. Parsed Logical Plan    — raw AST from your DataFrame/SQL code
#   2. Analyzed Logical Plan  — column names & types resolved vs catalog
#   3. Optimized Logical Plan — rules applied (pushdown, folding, pruning…)
#   4. Physical Plan          — execution strategy chosen (join algo, Exchange…)
# ═══════════════════════════════════════════════════════════════════════════ #
section("7. ALL FOUR CATALYST PLAN STAGES")
print("""
  Use explain("extended") to see every stage in one shot.
  Compare the Optimized Logical Plan against the Parsed/Analyzed plans
  to see exactly what rules fired.
""")

(
    spark.range(1_000)
    .withColumn("dept", (F.col("id") % 5).cast("int"))
    .filter(F.col("dept") == 3)
    .groupBy("dept")
    .agg(F.count("*").alias("cnt"))
    .explain("extended")
)



  7. ALL FOUR CATALYST PLAN STAGES


  Use explain("extended") to see every stage in one shot.
  Compare the Optimized Logical Plan against the Parsed/Analyzed plans
  to see exactly what rules fired.

== Parsed Logical Plan ==
'Aggregate ['dept], ['dept, 'count(*) AS cnt#44]
+- Filter (dept#43 = 3)
   +- Project [id#42L, cast((id#42L % cast(5 as bigint)) as int) AS dept#43]
      +- Range (0, 1000, step=1, splits=Some(10))

== Analyzed Logical Plan ==
dept: int, cnt: bigint
Aggregate [dept#43], [dept#43, count(1) AS cnt#44L]
+- Filter (dept#43 = 3)
   +- Project [id#42L, cast((id#42L % cast(5 as bigint)) as int) AS dept#43]
      +- Range (0, 1000, step=1, splits=Some(10))

== Optimized Logical Plan ==
Aggregate [dept#43], [dept#43, count(1) AS cnt#44L]
+- Project [cast((id#42L % 5) as int) AS dept#43]
   +- Filter (cast((id#42L % 5) as int) = 3)
      +- Range (0, 1000, step=1, splits=Some(10))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[dept#43],

In [21]:

# ═══════════════════════════════════════════════════════════════════════════ #
# 8. COST-BASED OPTIMIZER (CBO) — join reordering
#
#  With table statistics collected, CBO can reorder multi-way joins to
#  minimise intermediate result sizes, rather than following the order
#  you wrote in code.
# ═══════════════════════════════════════════════════════════════════════════ #
section("8. COST-BASED OPTIMIZER — join reordering")
print("""
  CBO is disabled by default.  We enable it, ANALYZE the tables, then check
  whether the optimizer reorders a 3-way join written as big→med→small.

  Look for:  Statistics(sizeInBytes=…, rowCount=…) in the Optimized plan
             and a join order that differs from the one we wrote.
""")

spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "false")

for t in ("big", "med", "small"):
    spark.sql(f"DROP TABLE IF EXISTS {t}")

big   = spark.range(100_000).withColumnRenamed("id", "big_id")
med   = spark.range(1_000).withColumnRenamed("id", "med_id")
small = spark.range(10).withColumnRenamed("id", "small_id")

big.write.saveAsTable("big")
med.write.saveAsTable("med")
small.write.saveAsTable("small")

spark.sql("ANALYZE TABLE big   COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE med   COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE small COMPUTE STATISTICS FOR ALL COLUMNS")

for t in ("big", "med", "small"):
    row = spark.sql(f"DESCRIBE EXTENDED {t}") \
               .filter(F.col("col_name") == "Statistics") \
               .select("data_type").first()
    print(f"{t}: {row[0] if row else 'no stats'}")

spark.sql("DESCRIBE EXTENDED big").show(truncate=False)

big_t   = spark.table("big")
med_t   = spark.table("med")
small_t = spark.table("small")

# Written big→med→small; CBO should reorder to start with the smallest tables
(
    big_t
    .join(med_t,   big_t.big_id  == med_t.med_id)
    .join(small_t, med_t.med_id  == small_t.small_id)
    .explain("extended")
)

spark.conf.set("spark.sql.cbo.enabled", "false")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled", "true")


  8. COST-BASED OPTIMIZER — join reordering


  CBO is disabled by default.  We enable it, ANALYZE the tables, then check
  whether the optimizer reorders a 3-way join written as big→med→small.

  Look for:  Statistics(sizeInBytes=…, rowCount=…) in the Optimized plan
             and a join order that differs from the one we wrote.

big: 405578 bytes, 100000 rows
med: 9178 bytes, 1000 rows
small: 5039 bytes, 10 rows
+----------------------------+----------------------------------------+-------+
|col_name                    |data_type                               |comment|
+----------------------------+----------------------------------------+-------+
|big_id                      |bigint                                  |NULL   |
|                            |                                        |       |
|# Detailed Table Information|                                        |       |
|Catalog                     |spark_catalog                           |       |
|Database          

In [ ]:

# ─── done ─────────────────────────────────────────────────────────────────────
spark.stop()
